# 08 · Train with gradient descent

## What you will build

The training loop that turns your random network into a working classifier: a gradient-descent update, shuffled mini-batches, one training epoch, and a `train` function that records training and validation curves. At the end your own network, every piece of which you typed in notebooks 02–07, learns the three-class spirals.

## Learning objectives

- Apply the update rules $W_{t+1} = W_t - \eta\,\partial L/\partial W$ and $b_{t+1} = b_t - \eta\,\partial L/\partial b$.
- Split an epoch into shuffled mini-batches and count the updates it makes.
- Read training and validation loss curves.
- Choose a learning rate, and recognise one that is too small or diverges.

## Dependencies

Tier 1: NumPy and Matplotlib only.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# From notebook 07: backpropagation.

def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy: (P - Y) / n."""
    return (probs - np.eye(probs.shape[1])[y]) / len(y)


def dense_backward(dZ, A_prev, W):
    """Gradients of a dense layer: (dW, db, dA_prev)."""
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T


def backward(network, cache, probs, y):
    """Every layer's gradients, last layer first, returned in layer order."""
    grads = [None] * len(network)
    dZ = softmax_cross_entropy_grad(probs, y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
        grads[i] = {"W": dW, "b": db}
    return grads

## 1 · One step downhill

The gradient points in the direction in which the loss grows fastest. Gradient descent moves every parameter a small step the other way:

$$W_{t+1} = W_t - \eta \frac{\partial L}{\partial W}, \qquad b_{t+1} = b_t - \eta \frac{\partial L}{\partial b},$$

where $\eta$, the **learning rate**, sets the step size. Here the update returns a *new* network rather than changing the old one in place, which makes it easy to compare before and after.

### ✍️ Your turn: `sgd_step`

Return a new list of layers where each `W` becomes `W - lr * dW` and each `b` becomes `b - lr * db`. `grads` has the same structure as `network`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def sgd_step_reference(network, grads, lr):
    return [{"W": layer["W"] - lr * g["W"], "b": layer["b"] - lr * g["b"]} for layer, g in zip(network, grads)]

In [ ]:
def sgd_step(network, grads, lr):
    """A new network, one gradient-descent step downhill."""
    # ✍️ Your code here (a list comprehension over zip(network, grads))
    raise NotImplementedError

In [ ]:
net_check = init_network([2, 3, 2], np.random.default_rng(0))
grads_check = [{"W": np.ones_like(l["W"]), "b": np.full_like(l["b"], 2.0)} for l in net_check]
sgd_step = practice("sgd_step", sgd_step, sgd_step_reference, cases=[
    (net_check, grads_check, 0.1),
    (net_check, grads_check, 0.0),
])

### Does one step help?

Take one full-batch step on the spirals and compare the loss before and after. With a sensible learning rate it goes down, a little.

In [ ]:
X, y = make_spirals(150, 3, 0.2, np.random.default_rng(1))
network = init_network([2, 64, 3], np.random.default_rng(2))
probs, cache = forward(network, X)
before = cross_entropy(probs, y)
stepped = sgd_step(network, backward(network, cache, probs, y), lr=0.5)
after = cross_entropy(forward(stepped, X)[0], y)
print(f"loss before {before:.4f}, after one step {after:.4f}")

## 2 · Mini-batches

Computing the gradient on all the data for every step is accurate but slow. **Mini-batch** gradient descent shuffles the examples each epoch and takes one step per small batch. The gradient is noisier, but you get many more updates per pass, and the noise even helps escape poor regions. One **epoch** is one pass over the data; with $n$ examples and batch size $B$ it makes $\lceil n / B \rceil$ updates.

### ✍️ Your turn: `minibatch_indices`

Return a list of index arrays that cover `range(n)` exactly once, in a shuffled order, in chunks of `batch_size` (the last chunk may be smaller). Shuffle with `np.random.default_rng(seed).permutation(n)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def minibatch_indices_reference(n, batch_size, seed):
    order = np.random.default_rng(seed).permutation(n)
    return [order[start:start + batch_size] for start in range(0, n, batch_size)]

In [ ]:
def minibatch_indices(n, batch_size, seed):
    """Shuffled batches of indices covering range(n)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
minibatch_indices = practice("minibatch_indices", minibatch_indices, minibatch_indices_reference, cases=[
    (10, 4, 0), (12, 3, 7), (5, 10, 1),
])
print("updates per epoch for 450 examples, batch 32:", len(minibatch_indices(450, 32, 0)))

## 3 · One epoch

An epoch puts everything together: for each mini-batch, run `forward`, measure the loss, run `backward`, and take an `sgd_step`. Every function in this loop is one you built.

### ✍️ Your turn: `train_epoch`

For each batch from `minibatch_indices(len(X), batch_size, seed)`: forward on `X[batch]`, record `cross_entropy(probs, y[batch])`, compute `backward`, and replace `network` with `sgd_step(network, grads, lr)`. Return `(network, mean of the recorded losses)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
def train_epoch_reference(network, X, y, lr, batch_size, seed):
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        network = sgd_step(network, backward(network, cache, probs, y[batch]), lr)
    return network, float(np.mean(losses))

In [ ]:
def train_epoch(network, X, y, lr, batch_size, seed):
    """One shuffled pass over the data. Returns (network, mean batch loss)."""
    losses = []
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
small = make_spirals(20, 3, 0.2, np.random.default_rng(3))
train_epoch = practice("train_epoch", train_epoch, train_epoch_reference, cases=[
    (init_network([2, 8, 3], np.random.default_rng(4)), small[0], small[1], 0.5, 16, 0),
])

## 4 · The training loop, with validation

Keep part of the data out of training: the **validation** set tells you whether the network is learning something general or just memorising. `train` runs epochs and records loss and accuracy on both sets after each one. (It is given here; it only calls functions you wrote.)

In [ ]:
def train(network, X, y, X_val, y_val, lr, epochs, batch_size, seed=0):
    """Train for some epochs; returns (network, history)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch(network, X, y, lr, batch_size, seed + epoch)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history


X_all, y_all = make_spirals(200, 3, 0.2, np.random.default_rng(10))
order = np.random.default_rng(11).permutation(len(X_all))
train_rows, val_rows = order[:480], order[480:]
X_train, y_train, X_val, y_val = X_all[train_rows], y_all[train_rows], X_all[val_rows], y_all[val_rows]

learning_rate = 1.0  # 🧪 try 0.05, 1.0 and 8.0, then run this cell and the plots again
network, history = train(init_network([2, 64, 3], np.random.default_rng(12)),
                         X_train, y_train, X_val, y_val, lr=learning_rate, epochs=120, batch_size=32)
print(f"final training accuracy {history['train_acc'][-1]:.3f}, validation accuracy {history['val_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(history["train_loss"], label="training"); axes[0].plot(history["val_loss"], label="validation")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].legend()
axes[1].plot(history["train_acc"], label="training"); axes[1].plot(history["val_acc"], label="validation")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
grid = np.linspace(-1.1, 1.1, 150)
gx, gy = np.meshgrid(grid, grid)
regions = predict(network, np.column_stack([gx.ravel(), gy.ravel()])).reshape(gx.shape)
plt.figure(figsize=(4.5, 4.5))
plt.contourf(gx, gy, regions, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis", alpha=0.35)
plt.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap="viridis", s=14, edgecolors="k", linewidths=0.3)
plt.title("your trained network: the regions follow the spirals")
plt.show()

## 5 · 🧪 Learning rates: too small, about right, too large

The same network, trained for 40 epochs with three learning rates. Too small and it barely moves; too large and the loss jumps around or blows up to `inf`/`nan`, which `np.isfinite` detects.

In [ ]:
for lr in (0.02, 1.0, 50.0):
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        _, h = train(init_network([2, 64, 3], np.random.default_rng(12)), X_train, y_train, X_val, y_val,
                     lr=lr, epochs=40, batch_size=32)
    final = h["train_loss"][-1]
    status = "diverged" if not np.isfinite(final) or final > h["train_loss"][0] else "ok"
    print(f"lr={lr:<5}: first loss {h['train_loss'][0]:.3f}, last loss {final:.3f}, val acc {h['val_acc'][-1]:.2f} ({status})")

## 6 · Batch, stochastic and mini-batch descent

The batch size decides how many updates an epoch makes and how noisy each one is: `batch_size = len(X)` is full-batch descent (1 update per epoch), `batch_size = 1` is stochastic gradient descent (one update per example), and anything in between is mini-batch descent.

In [ ]:
for batch_size in (len(X_train), 32, 1):
    updates = len(minibatch_indices(len(X_train), batch_size, 0))
    print(f"batch size {batch_size:3d}: {updates:3d} updates per epoch")

## 7 · Put it together

In [ ]:
assert history["train_loss"][-1] < 0.5 * history["train_loss"][0], "training should cut the loss by more than half"
assert history["val_acc"][-1] >= 0.85, f"expected at least 0.85 validation accuracy, got {history['val_acc'][-1]:.3f}"
print(f"✅ Your network learned the spirals: validation accuracy {history['val_acc'][-1]:.3f}.")

## Exercises

1. **Momentum.** Write `momentum_step(network, grads, velocity, lr, beta=0.9)` that keeps a running velocity $v \leftarrow \beta v - \eta\, g$ and updates $W \leftarrow W + v$. Does it reach 0.85 validation accuracy in fewer epochs?
2. **Learning-rate decay.** Multiply the learning rate by 0.98 after every epoch. Compare the final validation accuracy with a constant rate.
3. **Width.** Train `[2, 8, 3]` and `[2, 256, 3]`. Which underfits? Does the bigger one overfit on 480 examples?
4. **Open question.** The validation set chose nothing yet: we only watched it. In notebook 12 you will use it to pick the epoch to keep. Why must the *test* set stay out of that decision?

## Common errors and debugging

- **Loss is `nan`**: the learning rate is too large; divide it by 10.
- **Loss never moves**: you forgot to use the network returned by `sgd_step`.
- **Validation accuracy far below training accuracy**: the model is memorising; use more data, a smaller network or fewer epochs.

## Summary

Training repeats four steps on shuffled mini-batches: forward, loss, backward, update. The learning rate sets the step size: too small and nothing happens, too large and it diverges. Watching a held-out validation set tells you whether the network is learning something general. Your network, built entirely from your own functions, now separates the spirals.

## Next

**12 · Final project**: package everything into your own `NeuralNetwork` class, train it with a proper train/validation/test protocol, save it and reload it.